# Scenario Analysis and Risk Management

## Introduction

This notebook demonstrates how to perform **scenario analysis** on an options portfolio using realistic market data:

1. **Building** a portfolio with realistic term structures and vol surfaces
2. **Defining** scenario shocks (spot, vol, rates)
3. **Running** scenarios and computing P&L impact
4. **Visualizing** risk profiles

---

### What is Scenario Analysis?

Scenario analysis answers: **"What would happen to my portfolio if...?"**

| Scenario Type | Example | Purpose |
|---------------|---------|----------|
| **Spot shock** | EUR/USD moves ±5% | Test directional risk |
| **Vol shock** | Implied vol jumps +10 points | Test vega exposure |
| **Rate shock** | Yield curve shifts +50bp | Test rate sensitivity |
| **Historical** | Replay 2008 crisis | Test tail risk |

In [ ]:
# =============================================================================
# SETUP: Imports
# =============================================================================

import sys
from pathlib import Path
from datetime import date
import numpy as np

sys.path.insert(0, str(Path.cwd().parents[1]))

# Plotting
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# QuantStrata imports
from src.marketdata.core.market import Market
from src.marketdata.core.ids import MarketId
from src.marketdata.core.interfaces import Quote
from src.marketdata.curves.term_structure import ZeroRateCurve
from src.marketdata.surfaces.vol_surface import GridVolSurface
from src.portfolio.core import Portfolio, Position
from src.instruments.fx.options.vanilla import FxVanillaEuropeanOption
from src.pricers.fx.european_bsm import FxVanillaEuropeanOptionBsmPricer

print("All imports successful!")

## 1. Building Realistic Market Data

In [ ]:
# =============================================================================
# Define Realistic Market Data
# =============================================================================

# Market IDs
spot_id = MarketId.parse("FX.SPOT.EURUSD")
vol_id = MarketId.parse("FX.VOL.EURUSD")
usd_curve_id = MarketId.parse("IR.ZERO.USD")
eur_curve_id = MarketId.parse("IR.ZERO.EUR")

# Base market parameters
BASE_SPOT = 1.0850

# Realistic term structures
TENORS = np.array([0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0])
USD_RATES = np.array([0.0540, 0.0535, 0.0520, 0.0480, 0.0460, 0.0440, 0.0435, 0.0430, 0.0425, 0.0420])
EUR_RATES = np.array([0.0380, 0.0385, 0.0390, 0.0395, 0.0400, 0.0405, 0.0408, 0.0410, 0.0412, 0.0415])

# Realistic vol surface
VOL_EXPIRIES = np.array([0.083, 0.167, 0.25, 0.5, 1.0])
VOL_STRIKES = np.array([0.95, 0.98, 1.00, 1.02, 1.05, 1.08, 1.10, 1.12, 1.15]) * BASE_SPOT
VOL_GRID = np.array([
    [0.115, 0.100, 0.088, 0.085, 0.090, 0.095, 0.100, 0.108, 0.118],
    [0.112, 0.098, 0.087, 0.084, 0.088, 0.093, 0.098, 0.105, 0.115],
    [0.110, 0.096, 0.086, 0.084, 0.087, 0.092, 0.096, 0.103, 0.112],
    [0.108, 0.095, 0.086, 0.085, 0.088, 0.092, 0.095, 0.100, 0.108],
    [0.105, 0.094, 0.088, 0.087, 0.090, 0.093, 0.096, 0.100, 0.106],
])

def create_market(spot=BASE_SPOT, vol_shift=0.0, rate_shift=0.0):
    """Create market with optional shocks."""
    return Market(
        asof=date.today(),
        quotes={spot_id: Quote(value=spot)},
        curves={
            usd_curve_id: ZeroRateCurve(tenors=TENORS, zero_rates=USD_RATES + rate_shift),
            eur_curve_id: ZeroRateCurve(tenors=TENORS, zero_rates=EUR_RATES + rate_shift),
        },
        vols={
            vol_id: GridVolSurface(
                expiries=VOL_EXPIRIES,
                strikes=VOL_STRIKES,
                implied_vols=np.maximum(0.01, VOL_GRID + vol_shift),
            ),
        },
    )

base_market = create_market()

print("Base Market Data:")
print("="*50)
print(f"  EURUSD Spot:     {BASE_SPOT:.4f}")
print(f"  USD 3M rate:     {USD_RATES[0]:.2%}")
print(f"  EUR 3M rate:     {EUR_RATES[0]:.2%}")
print(f"  3M ATM vol:      {VOL_GRID[2, 4]:.2%}")
print(f"  Vol surface:     {VOL_GRID.shape[0]} expiries x {VOL_GRID.shape[1]} strikes")

## 2. Building a Portfolio

In [ ]:
# =============================================================================
# Build Portfolio
# =============================================================================

positions = []

# Position 1: Long EURUSD Call (bullish directional bet)
positions.append(Position(
    position_id="EURUSD_CALL_OTM",
    instrument=FxVanillaEuropeanOption(
        option_type="call",
        notional=10_000_000,
        strike=1.10,
        expiry=0.25,
        spot_id=spot_id,
        vol_id=vol_id,
        domestic_curve_id=usd_curve_id,
        foreign_curve_id=eur_curve_id,
    ),
    quantity=1,
))

# Position 2: Short EURUSD Put (selling downside protection)
positions.append(Position(
    position_id="EURUSD_PUT_SHORT",
    instrument=FxVanillaEuropeanOption(
        option_type="put",
        notional=10_000_000,
        strike=1.05,
        expiry=0.25,
        spot_id=spot_id,
        vol_id=vol_id,
        domestic_curve_id=usd_curve_id,
        foreign_curve_id=eur_curve_id,
    ),
    quantity=-1,
))

# Position 3: Long Straddle (volatility play, 6-month)
positions.append(Position(
    position_id="EURUSD_STRADDLE_CALL",
    instrument=FxVanillaEuropeanOption(
        option_type="call",
        notional=5_000_000,
        strike=1.085,
        expiry=0.5,
        spot_id=spot_id,
        vol_id=vol_id,
        domestic_curve_id=usd_curve_id,
        foreign_curve_id=eur_curve_id,
    ),
    quantity=1,
))

positions.append(Position(
    position_id="EURUSD_STRADDLE_PUT",
    instrument=FxVanillaEuropeanOption(
        option_type="put",
        notional=5_000_000,
        strike=1.085,
        expiry=0.5,
        spot_id=spot_id,
        vol_id=vol_id,
        domestic_curve_id=usd_curve_id,
        foreign_curve_id=eur_curve_id,
    ),
    quantity=1,
))

portfolio = Portfolio(positions=positions)

print("\nPortfolio Summary:")
print("="*70)
for pos in portfolio:
    inst = pos.instrument
    direction = "LONG" if pos.quantity > 0 else "SHORT"
    print(f"  {pos.position_id:<22} {direction:<6} {inst.option_type:<4} K={inst.strike:.4f} T={inst.expiry}Y")

In [ ]:
# =============================================================================
# Price Portfolio (Base Case)
# =============================================================================

pricer = FxVanillaEuropeanOptionBsmPricer()

def price_portfolio(market, portfolio):
    """Price all positions and return results."""
    results = {}
    totals = {"pv": 0, "delta": 0, "gamma": 0, "vega": 0, "theta": 0}
    
    for pos in portfolio:
        pv = pricer.price(pos.instrument, market) * pos.quantity
        greeks = pricer.greeks(pos.instrument, market)
        
        results[pos.position_id] = {
            "pv": pv,
            "delta": greeks.get("delta", 0) * pos.quantity,
            "gamma": greeks.get("gamma", 0) * pos.quantity,
            "vega": greeks.get("vega", 0) * pos.quantity,
            "theta": greeks.get("theta", 0) * pos.quantity,
        }
        
        for key in totals:
            totals[key] += results[pos.position_id][key]
    
    results["TOTAL"] = totals
    return results

base_results = price_portfolio(base_market, portfolio)

print("\nBase Case Pricing:")
print("="*80)
print(f"{'Position':<25} {'PV (USD)':<15} {'Delta':<12} {'Vega':<12} {'Theta':<12}")
print("-"*80)

for pos_id, values in base_results.items():
    print(f"{pos_id:<25} ${values['pv']:>12,.0f} {values['delta']:>12,.0f} "
          f"{values['vega']:>12,.0f} {values['theta']:>12,.0f}")

print(f"\nPortfolio PV: ${base_results['TOTAL']['pv']:,.0f}")

## 3. Running Scenario Analysis

In [ ]:
# =============================================================================
# Define and Run Scenarios
# =============================================================================

scenarios = {
    'Spot +1%':   {'spot_shock': 0.01, 'vol_shock': 0},
    'Spot +5%':   {'spot_shock': 0.05, 'vol_shock': 0},
    'Spot -1%':   {'spot_shock': -0.01, 'vol_shock': 0},
    'Spot -5%':   {'spot_shock': -0.05, 'vol_shock': 0},
    'Vol +2pts':  {'spot_shock': 0, 'vol_shock': 0.02},
    'Vol +5pts':  {'spot_shock': 0, 'vol_shock': 0.05},
    'Vol -2pts':  {'spot_shock': 0, 'vol_shock': -0.02},
    'Risk-Off':   {'spot_shock': -0.05, 'vol_shock': 0.05},
    'Risk-On':    {'spot_shock': 0.05, 'vol_shock': -0.02},
}

base_pv = base_results['TOTAL']['pv']
scenario_results = {}

for name, params in scenarios.items():
    shocked_market = create_market(
        spot=BASE_SPOT * (1 + params['spot_shock']),
        vol_shift=params['vol_shock'],
    )
    results = price_portfolio(shocked_market, portfolio)
    pnl = results['TOTAL']['pv'] - base_pv
    
    scenario_results[name] = {
        'pv': results['TOTAL']['pv'],
        'pnl': pnl,
        'pnl_pct': pnl / abs(base_pv) * 100 if base_pv != 0 else 0,
    }

print("\nScenario Analysis Results:")
print("="*70)
print(f"Base PV: ${base_pv:,.0f}")
print("-"*70)
print(f"{'Scenario':<20} {'Scenario PV':>15} {'P&L':>15} {'P&L %':>10}")
print("-"*70)

for name, result in scenario_results.items():
    print(f"{name:<20} ${result['pv']:>13,.0f} ${result['pnl']:>+13,.0f} {result['pnl_pct']:>+9.1f}%")

## 4. Visualizing Results

In [ ]:
# =============================================================================
# Visualize Scenario P&L
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: P&L by Scenario
ax1 = axes[0]
sorted_scenarios = sorted(scenario_results.items(), key=lambda x: x[1]['pnl'])
names = [s[0] for s in sorted_scenarios]
pnls = [s[1]['pnl'] / 1000 for s in sorted_scenarios]
colors = ['red' if p < 0 else 'green' for p in pnls]

ax1.barh(names, pnls, color=colors, alpha=0.7, edgecolor='black')
ax1.axvline(x=0, color='black', linewidth=1)
ax1.set_xlabel('P&L (USD thousands)')
ax1.set_title('Scenario P&L Impact', fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Plot 2: Spot Sensitivity
ax2 = axes[1]
spot_range = np.linspace(BASE_SPOT * 0.90, BASE_SPOT * 1.10, 30)
spot_pnls = []

for spot in spot_range:
    shocked_market = create_market(spot=spot)
    results = price_portfolio(shocked_market, portfolio)
    spot_pnls.append((results['TOTAL']['pv'] - base_pv) / 1000)

ax2.plot(spot_range, spot_pnls, 'b-', linewidth=2)
ax2.fill_between(spot_range, spot_pnls, 0, where=(np.array(spot_pnls) > 0), alpha=0.3, color='green')
ax2.fill_between(spot_range, spot_pnls, 0, where=(np.array(spot_pnls) < 0), alpha=0.3, color='red')
ax2.axhline(y=0, color='black', linewidth=1)
ax2.axvline(x=BASE_SPOT, color='gray', linestyle='--', alpha=0.7)
ax2.set_xlabel('EURUSD Spot')
ax2.set_ylabel('P&L (USD thousands)')
ax2.set_title('Portfolio P&L vs Spot Price', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Vol Sensitivity
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 5))

vol_shifts = np.linspace(-0.03, 0.08, 30)
vol_pnls = []

for vol_shift in vol_shifts:
    shocked_market = create_market(vol_shift=vol_shift)
    results = price_portfolio(shocked_market, portfolio)
    vol_pnls.append((results['TOTAL']['pv'] - base_pv) / 1000)

ax.plot(vol_shifts * 100, vol_pnls, 'purple', linewidth=2)
ax.fill_between(vol_shifts * 100, vol_pnls, 0, where=(np.array(vol_pnls) > 0), alpha=0.3, color='green')
ax.fill_between(vol_shifts * 100, vol_pnls, 0, where=(np.array(vol_pnls) < 0), alpha=0.3, color='red')
ax.axhline(y=0, color='black', linewidth=1)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Vol Shift (percentage points)')
ax.set_ylabel('P&L (USD thousands)')
ax.set_title('Portfolio P&L vs Volatility Shift', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Portfolio Vega: ${base_results['TOTAL']['vega']:,.0f} per vol point")
if base_results['TOTAL']['vega'] > 0:
    print("Portfolio is LONG volatility (benefits from vol increase)")
else:
    print("Portfolio is SHORT volatility (benefits from vol decrease)")

## 5. Key Takeaways

### Realistic Market Data
- Term structures affect pricing at different expiries differently
- Vol surfaces capture smile effects that flat vol misses
- Scenarios should shift the full surface, not just a single number

### Scenario Analysis
- **Largest loss**: Identify worst-case for risk limits
- **Asymmetry**: Options create non-linear payoffs
- **Combined shocks**: Risk-off scenarios (spot down + vol up) often occur together